<a href="https://colab.research.google.com/github/vaishnavi2810-code/AI-For-Beginners/blob/main/reviews.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Predicting Sentiment Labels

In this experiment, you will explore the accuracy of sentiment classificaiton using different feature representations of text documents.

First, you will implement `createBasicFeatures`, which creates a sparse matrix representation of a collection of documents.  As we discussed in class, many classification models represent a document as a sparse vector of of features and their weights. Weights could be boolean, counts, tf-idf values, or other functions of the document. In a linear model, we then combine the document vectors with the candidate output classes and learn weights for each of the feature-class pairs. This last step is implemented for you.

For this exercise, you should have a feature for each word containing at least one alphabetic character. You may use the `numpy` and `sklearn` packages to help with implementing a sparse matrix.

Then, you will implement `createFancyFeatures`, which can specify at any other features you choose to help improve performance on the classification task.

The two code blocks at the end train and evaluate two models—logistic regression with L1 and L2 regularization—using your featurization functions. Besides held-out classification accuracy with 10-fold cross-validation, you will also see the features in each class given high weights by the model.

There are many helpful resources online for getting up to speed with vector representations of documents. One example is the first two chapters of Delip Rao and Brian McMahan, _Natural Language Processing with PyTorch_, O'Reilly, 2019.  You should be able to <a href="https://learning.oreilly.com/library/view/natural-language-processing/9781491978221/">read it online</a> via the Northeastern Library's subscription using a <tt>northeastern.edu</tt> email address.

In [ ]:
import json
import requests
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_validate,LeaveOneOut,KFold
import numpy as np

In [ ]:
# read in the movie review corpus
def readReviews():
  raw = requests.get("https://raw.githubusercontent.com/dasmiq/cs6120-assignment2/refs/heads/main/cornell_reviews.json").text.strip()
  corpus = [json.loads(line) for line in raw.split("\n")]

  return corpus

This is where you will implement two functions to featurize the data.

In [ ]:
# TODO: Implement createBasicFeatures
# NB: The current contents are for testing only
# This function should return:
#  -a sparse numpy matrix of document features
#  -a list of the correct class for each document
#  -a list of the vocabulary used by the features, such that the ith term of the
#    list is the word whose counts appear in the ith column of the matrix.

# This function should create a feature representation using all tokens that
# contain an alphabetic character.

from sklearn.feature_extraction.text import CountVectorizer
import re
def createBasicFeatures(corpus):
  #Your code here
  texts = [doc["text"] for doc in corpus]
  classes = [doc["class"] for doc in corpus]

  # Define token pattern: keep tokens with at least one alphabetic character
  token_pattern = r"(?u)\b\w*[A-Za-z]\w*\b"

  # Use CountVectorizer to build the matrix
  vectorizer = CountVectorizer(token_pattern=token_pattern, lowercase=True)
  X = vectorizer.fit_transform(texts)

  # Get vocab list in the correct column order. It tells us which word is there corresponding to each column.
  vocab = vectorizer.get_feature_names_out().tolist()
  return X,classes,vocab

In [ ]:
# TODO: Implement createFancyFeatures andn describe in comments what the
# features are and why they might ben helpful.
# This function can add other features you want that help classification
# accuracy, such as bigrams, word prefixes and suffixes, etc.
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from scipy.sparse import hstack

# Example sentiment lexicons (can be expanded)
POSITIVE_WORDS = {"good", "great", "excellent", "amazing", "love", "enjoyed", "fantastic"}
NEGATIVE_WORDS = {"bad", "terrible", "awful", "hate", "boring", "worst", "disappointing"}
def createFancyFeatures(corpus):
  #Your code here
  # Extract texts and labels
  texts = [doc["text"] for doc in corpus]
  classes = [doc["class"] for doc in corpus]

  # --- (1) TF-IDF with unigrams + bigrams + trigrams: These capture single words, common word pairs and triplets.
  # This helps the model to analyze choice of words, the context patterns and phrases in the text which could be useful features for the model
  token_pattern = r"(?u)\b\w*[A-Za-z]\w*\b"
  vectorizer = TfidfVectorizer(token_pattern=token_pattern, lowercase=True, ngram_range=(1, 3), min_df=2)
  X_tfidf = vectorizer.fit_transform(texts)

  # --- (2) Extra handcrafted features: The prefix and suffix takes the first 3 and last 3 letters of each word.
  # It helps in detecting common morphological patterns. For example: words starting with 'dis' are most likely negative: dislike, diasater
  # Also words ending with 'ful' are mostly likely positive: joyful. This is how it helps in identifying patterns.
  prefix_features = [" ".join([w[:3] for w in t.split()]) for t in texts]
  suffix_features = [" ".join([w[-3:] for w in t.split()]) for t in texts]

  # --- (3) All-caps word count: Uppercase words are most like indicating of strong emotions. It can be either positive or negative which can be correlated to sentiment.
  # For example: 'AWESOME', 'TERRIBLE'. Strong emotions can also help in determining either positive or negative of the given text.
  all_caps_count = np.array([sum(1 for w in t.split() if w.isupper()) for t in texts]).reshape(-1, 1)

  # --- (4) Sentiment lexicon counts : pos_count is counts words from a positive sentiment lexicon.
  # neg_count is counts words from a negative sentiment lexicon.
  # These are direct sentiment cues, helping the model identify overall positive or negative tone regardless of word form.
  pos_count = np.array([sum(1 for w in t.lower().split() if w in POSITIVE_WORDS) for t in texts]).reshape(-1, 1)
  neg_count = np.array([sum(1 for w in t.lower().split() if w in NEGATIVE_WORDS) for t in texts]).reshape(-1, 1)

  prefix_vectorizer = CountVectorizer(min_df = 2)
  suffix_vectorizer = CountVectorizer(min_df = 2)
  X_prefix = prefix_vectorizer.fit_transform(prefix_features)
  X_suffix = suffix_vectorizer.fit_transform(suffix_features)

  # --- (5) Document length may also correlate with sentiment intensity.
  # For example: Longer reviews might indicate more stronger details/intensity. These can also be important features to train the model.
  lengths = np.array([len(t.split()) for t in texts]).reshape(-1, 1)

  # --- (6) An exclaimation also indicates a strong intensity which may correlate to the sentiment.
  # For example: 'Loved It!!' or 'Hated It!'. This can also help the model in analyzing the sentiment better.
  exclaims = np.array([t.count("!") for t in texts]).reshape(-1, 1)

  # Combine all features into one sparse matrix
  X = hstack([X_tfidf, lengths, exclaims, all_caps_count, pos_count, neg_count, X_prefix, X_suffix])

  # --- Build vocabulary list ---
  vocab = vectorizer.get_feature_names_out().tolist()
  vocab += ["doc_length", "exclaim_count","all_caps_count","pos_count","neg_count"]
  vocab += ["prefix_" + f for f in prefix_vectorizer.get_feature_names_out()]
  vocab += ["suffix_" + f for f in suffix_vectorizer.get_feature_names_out()]
  return X,classes,vocab

In [ ]:
# given a numpy matrix representation of the features for the training set, the
# vector of true classes for each example, and the vocabulary as described
# above, this computes the accuracy of the model using leave one out cross
# validation and reports the most indicative features for each class

def evaluateModel(X,y,vocab,penalty="l1"):
  # create and fit the model
  model = LogisticRegression(penalty=penalty,solver="liblinear")
  results = cross_validate(model,X,y,cv=KFold(n_splits=10, shuffle=True, random_state=1))

  # determine the average accuracy
  scores = results["test_score"]
  avg_score = sum(scores)/len(scores)

  # determine the most informative features
  # this requires us to fit the model to everything, because we need a
  # single model to draw coefficients from, rather than 26
  model.fit(X,y)
  class0_weight_sorted = model.coef_[0, :].argsort()
  class1_weight_sorted = (-model.coef_[0, :]).argsort()

  termsToTake = 20
  class0_indicators = [vocab[i] for i in class0_weight_sorted[:termsToTake]]
  class1_indicators = [vocab[i] for i in class1_weight_sorted[:termsToTake]]

  if model.classes_[0] == "pos":
    return avg_score,class0_indicators,class1_indicators
  else:
    return avg_score,class1_indicators,class0_indicators

def runEvaluation(X,y,vocab):
  print("----------L1 Norm-----------")
  avg_score,pos_indicators,neg_indicators = evaluateModel(X,y,vocab,"l1")
  print("The model's average accuracy is %f"%avg_score)
  print("The most informative terms for pos are: %s"%pos_indicators)
  print("The most informative terms for neg are: %s"%neg_indicators)
  #this call will fit a model with L2 normalization
  print("----------L2 Norm-----------")
  avg_score,pos_indicators,neg_indicators = evaluateModel(X,y,vocab,"l2")
  print("The model's average accuracy is %f"%avg_score)
  print("The most informative terms for pos are: %s"%pos_indicators)
  print("The most informative terms for neg are: %s"%neg_indicators)

In [ ]:
corpus = readReviews()

Run the following to train and evaluate two models using basic features:

In [ ]:
X,y,vocab = createBasicFeatures(corpus)
runEvaluation(X, y, vocab)

----------L1 Norm-----------
The model's average accuracy is 0.827000
The most informative terms for pos are: ['flaws', 'memorable', 'terrific', 'edge', 'sherri', 'excellent', 'perfectly', 'masterpiece', 'enjoyable', 'using', 'fun', 'overall', 'gas', 'solid', 'command', 'quite', 'follows', 'fantastic', 'different', 'liar']
The most informative terms for neg are: ['waste', 'mess', 'ridiculous', 'lame', 'worst', 'headed', 'awful', 'unfortunately', 'cheap', 'write', 'boring', 'superior', 'tedious', 'bad', 'jesse', 'terrible', 'poor', 'maybe', 'looks', 'jakob']
----------L2 Norm-----------
The model's average accuracy is 0.833000
The most informative terms for pos are: ['fun', 'great', 'back', 'quite', 'well', 'excellent', 'overall', 'seen', 'american', 'perfectly', 'yet', 'memorable', 'terrific', 'job', 'pulp', 'true', 'very', 'performances', 'solid', 'different']
The most informative terms for neg are: ['bad', 'unfortunately', 'worst', 'waste', 'nothing', 'only', 'script', 'boring', 'awf

Run the following to train and evaluate two models using extended features:

In [ ]:
X,y,vocab = createFancyFeatures(corpus)
runEvaluation(X, y, vocab)

----------L1 Norm-----------
The model's average accuracy is 0.795500
The most informative terms for pos are: ['suffix_rue', 'prefix_edg', 'suffix_yet', 'prefix_196', 'prefix_sky', 'suffix_fun', 'prefix_enj', 'suffix_997', 'suffix_kie', 'suffix_til', 'suffix_zel', 'prefix_flo', 'pos_count', 'prefix_sur', 'suffix_fic', 'suffix_rks', 'prefix_esp', 'prefix_due', 'suffix_lar', 'suffix_job']
The most informative terms for neg are: ['neg_count', 'prefix_poo', 'prefix_dut', 'prefix_vik', 'prefix_tir', 'prefix_pot', 'suffix_olf', 'suffix_lat', 'prefix_dol', 'suffix_eff', 'prefix_unf', 'prefix_dig', 'suffix_lve', 'prefix_tal', 'prefix_tv', 'prefix_uni', 'prefix_lam', 'prefix_mes', 'prefix_bar', 'prefix_bor']
----------L2 Norm-----------
The model's average accuracy is 0.801500
The most informative terms for pos are: ['suffix_fun', 'pos_count', 'prefix_enj', 'suffix_ces', 'suffix_kie', 'prefix_won', 'prefix_bac', 'prefix_sur', 'suffix_rue', 'prefix_196', 'suffix_lar', 'suffix_you', 'prefix_del',

**TODO**: Briefly comment on your results. You do _not_ need to beat the basic features model to get full credit.

1. Basic Features Model: The classifier with the highest accuracy (L1 ≈ 82.7%, L2 = 83.3%) uses simple bag-of-words characteristics.
The most informative words are positive adjectives like "great," "fun," "memorable," "terrific," and "masterpiece". The negative words are like "waste," "boring," "awful," and "ridiculous". Since they are the words that inherently express either positive or negative sentiment, these characteristics are directly interpretable. This suggests that since the sentiment is stated openly through certain phrases, raw word occurrences are highly good at differentiating sentiment in movie reviews.
2. Fancy Features Model: The accuracy significantly decreases (L1 ≈ 79.5%, L2 ≈ 80.2%) when we include engineered features such as prefixes, suffixes, and positive/negative word counts. The top negative features are 'neg_count', 'prefix_poo', and 'prefix_pot', whereas the top positive features are'suffix_fun', 'prefix_enj', and 'pos_count'.These characteristics are more abstract since they record aggregated metrics (counts of positive or negative words) or patterns in word forms (prefixes/suffixes) rather than individual words. Even though the accuracy is less than the basic feature model, these words provide us more information and insights about the patterns. For example, positive terms like "enjoyable" or "funny" sometimes contain prefixes like "enj" or suffixes like "fun." Likewise, 'neg_count' combines several negative indicators in a document.
3. Interpretation and Takeaways:
Due to its reliance on sentiment words with high frequency, the simple model performs well for raw prediction. The fancy model is helpful for interpretability even though its accuracy is a little lower. Beyond specific words, it shows patterns like morphological clues and the general intensity of sentiment.This illustrates the typical machine learning trade-off between interpretability and accuracy: designed features help us understand why the model generates specific predictions, while simpler features may predict better. Essentially, these manufactured features are especially useful when comprehending the thinking behind the model is just as crucial as its actual prediction ability, such as when examining how language conveys sentiment across genres or historical texts.
4. L1 v/s L2 regularization: L1 is helpful for understanding which words most strongly convey sentiment and for providing a clear, comprehensible list of essential attributes. More subtle patterns are captured by L2, which may weaken interpretability but marginally enhance prediction.This clarifies the differences between your L1 and L2 feature lists: While L2 allocates weights more evenly, L1 zeros out a large number of coefficients in order to concentrate on the strongest signals.

